主要实现功能：整理库存数据

In [ ]:
import pandas as pd
from functools import reduce

import openpyxl
import warnings

pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

In [ ]:
date_parm = '20261116'

In [ ]:
fba_path = f'../src_data/FBA库存/FBA库存-自定义导出-{date_parm}.xlsx'

product_path = f'../src_data/产品库存/产品库存-自定义导出-{date_parm}.xlsx'

yuzhan_path = f'../src_data/预占单/调拨单20250325.xlsx'  # 积加上下载的已拣货未出库状态的调拨单

yiyuzhan_path = f'../src_data/预占单/未出库调拨单导出{date_parm}.xlsx'  # 积加上下载的未拣货未出库状态的调拨单

LogisticsMethod_map = pd.read_excel('../src_data/在途货件/预计到达.xlsx', sheet_name = '物流渠道映射')

yuzhan_df = pd.read_excel(yuzhan_path, sheet_name='明细数据', usecols=['SKU', 'MSKU', '调出数量', '调入仓', '单据状态', '拣货状态', '出库状态'])

In [ ]:
usecols_pro = ['仓库', '平台站点', 'SKU', 'MSKU',
                '采购量',  # 采购量  = 采购单未交货数量 = 已下单数量
                '在途量',  # 在途量 = 已发货量 + 待入库量 / 在途量 = 本地-在途 （待质检量）
                '可用量', '未处理预占', '已处理预占']  # 可用量 = 良品量 - 已处理预占量（已拣货，未出库的库存）        未处理预占（未拣货未出库的库存）    

product_df = pd.read_excel(io=product_path, usecols=usecols_pro)

In [ ]:
product_df['仓库'].unique()

In [ ]:
product_df.columns

In [ ]:
# 新增一列“已生产未发货”。(采购量)
# 由于'仓库'列值为'供应商成品本地仓'的行，采购量的值代表“已生产未发货”，则出现以下操作。

# 筛选出'仓库'列值为'供应商成品本地仓'的行，并取列'MSKU', '平台站点', 'SKU', '采购量'
product_df_factory = product_df[product_df['仓库'] == '供应商成品本地仓'][['MSKU', '平台站点', 'SKU', '采购量']]
# 重命名新列为你想要的名称
product_df_factory.rename(columns={'采购量': '已生产未发货'}, inplace=True)

# 合并表，得到新列'已生产未发货'
product_df_new = pd.merge(product_df, product_df_factory, on=['MSKU', '平台站点', 'SKU'], how='left')
# '仓库'列不包含"广州仓"的行，'已生产未发货'列赋值为0
product_df_new.loc[~product_df_new['仓库'].astype(str).str.contains('广州仓', na=False), '已生产未发货'] = 0
# '已生产未发货'列空值填充为0
product_df_new['已生产未发货'] = product_df_new['已生产未发货'].fillna(0)

In [ ]:
# 合并'在途量'
# 筛选出'仓库'列值为'供应商成品本地仓'的行，并取列'MSKU', '平台站点', 'SKU', '在途量'
product_df_transit = product_df[product_df['仓库'] == '供应商成品本地仓'][['MSKU', '平台站点', 'SKU', '在途量']]
# 重命名新列为你想要的名称
product_df_transit.rename(columns={'在途量': '在途量_工厂'}, inplace=True)

# 合并表，得到新列'在途量_工厂'
product_df = pd.merge(product_df_new, product_df_transit, on=['MSKU', '平台站点', 'SKU'], how='left')

# '仓库'列不包含"广州仓"的行，'在途量'列赋值为0
product_df.loc[~product_df['仓库'].astype(str).str.contains('广州仓', na=False), '在途量_工厂'] = 0

# '在途量'列空值填充为0
product_df['在途量_工厂'] = product_df['在途量_工厂'].fillna(0)

product_df['在途量'] = product_df['在途量_工厂'] + product_df['在途量']

In [ ]:
# product_df.to_excel('../src_data/产品库存/产品库存test5.xlsx', index = False)

In [ ]:
# 删除 仓库 列值为'供应商成品本地仓'的行（保留不等于'b'的行）
product_df = product_df[product_df['仓库'] != '供应商成品本地仓']
product_df = product_df.drop(columns=['在途量_工厂'])

In [ ]:
# product_df.to_excel('../src_data/产品库存/产品库存test6.xlsx', index = False)

In [ ]:
# 20251216 新增逻辑，预占总数等于=已处理预占（已拣货且未出库） + 未处理预占（未拣货且未出库）；
# 可用量原本就减掉了“已处理预占”，还需减掉“未处理预占”，这个预占总数都需要归为在途数，所以要在本地库存中扣减。
product_df['可用量'] = product_df['可用量'] - product_df['未处理预占']
product_df['预占总数'] = product_df['已处理预占'] + product_df['未处理预占']
product_df = product_df.drop(columns = ['已处理预占', '未处理预占'])

# product_df.rename(columns={'SKU0001': 'SKU', '采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '待入库量': '本地-待入库量'}, inplace=True)  # todo
product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

# 这段代码会找到所有“仓库”为 "TK本地仓" 且“站点”为 "共享" 的行，并将其“站点”值改为 "TIKTOK:TK本地仓:TK"
product_df.loc[(product_df['仓库'] == 'TK本地仓') & (product_df['站点'] == '共享'), '站点'] = 'TIKTOK:TK本地仓:TK本地仓'

In [ ]:
# usecols_fba = ['仓库', 'SKU', 'MSKU', 
#                 '已发货(数量)', '正在接收(数量)',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
#                 '预留-运营中心转运(数量)', '预留-运营中心正在处理(数量)', 'FBA可售(数量)', '可用库存(数量)',  # 可用库存(数量) = FBA可售(数量) + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量)
#                 '271~330(数量)', '331~365(数量)', '365+(数量)', 
#                 '7天日均', '15天日均', '30天日均']

# usecols_fba = ['仓库', 'SKU', 'MSKU', 
#                 '货件处理中', '货件已发货', '货件正在接收',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
#                 '预留-运营中心转运', '预留-运营中心正在处理', '本地可售', '可用量',  # 可用量 = FBM可售 + 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售
#                 '271~330', '331~365', '365+', 
#                 '平台7天日均', '平台15天日均', '平台30天日均']
usecols_fba = ['仓库', 'SKU', 'MSKU', 
                '货件处理中', '货件已发货', '货件正在接收',  # FBA在途 = 已发货(数量) + 正在接收(数量) + 预占（从发货单中取数）
                '运营中心转运', '预留-运营中心正在处理', '本地可售', '可用量',  # 可用量 = FBM可售 + 本地可售 + 预留-运营中心正在处理(数量) + 预留-运营中心转运(数量) 良品量=FBA本地可售
                '271~330', '331~365', '365+', 
                '平台7天日均', '平台15天日均', '平台30天日均']
fba_df = pd.read_excel(io=fba_path, usecols=usecols_fba)
fba_df['已发货(数量)'] = fba_df['货件处理中'] + fba_df['货件已发货']
fba_df.rename(columns={'平台7天日均': '7天日均', '平台15天日均': '15天日均', '平台30天日均': '30天日均'}, inplace=True) 
# fba_df.rename(columns={'货件已发货': '已发货(数量)', '本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)
# fba_df.rename(columns={'本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)'}, inplace=True)
fba_df.rename(columns={'本地可售': 'FBA可售', '可用量': '可用库存','货件正在接收':'正在接收(数量)','运营中心转运':'预留-运营中心转运'}, inplace=True)

In [ ]:
# product_df = pd.read_excel(io=product_path, usecols=usecols_pro)
# product_df.rename(columns={'采购量': '已下单数量', '在途量': '本地-在途', '可用量': '本地', '平台站点': '站点'}, inplace=True)  # todo

In [ ]:
# product_df = new_product_df

In [ ]:
product_df.columns

# 1、预处理FBA库存数据

In [ ]:
# 去掉二次销售MSKU和MX站点的数据
fba_process = fba_df[~(fba_df.MSKU.str.contains('amzn') | fba_df.仓库.str.contains('MX_FBA'))]

# 格式化仓库 shop:site
fba_process['仓库'] = fba_process.仓库.str.split('_FBA', expand=True)[0]

# 把EU站点映射成DE
fba_process['仓库'] = fba_process['仓库'].apply(lambda x: x[:-2] + 'DE' if 'EU' in x else x)

# 获取站点
fba_process['站点'] = fba_process.仓库.str[-2:]

# 定义 长期仓储库存
fba_process['长期仓储库存'] = fba_process[['271~330', '331~365', '365+']].sum(axis=1)
fba_process.drop(columns=['271~330', '331~365', '365+'], inplace=True)

fba_process.query('SKU == "SP001-406 Black 40"')

In [ ]:
fba_process.columns

In [ ]:
fba_process.仓库.unique()

In [ ]:
fba_process.站点.unique()

# 2、预处理海外仓库存和本地仓库存

In [ ]:
# # 新增 站点 列，根据 仓库 列是否包含 "FBA" 来赋值
# product_df['站点'] = product_df['仓库'].apply(lambda x: x if 'FBA' in str(x) else '共享').str.replace('_FBA', '', regex=False)
# # 新增 店铺 列
# product_df['店铺'] = product_df['站点'].str.replace('EU', 'DE', regex=False)
# # 构建 站点
# product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x).str.replace('EU', 'DE', regex=False)

In [ ]:
product_df.仓库.unique()

In [ ]:
'''
# todo：枚举法排除
# 去除TIKTOK数据
product_df = product_df[~product_df['站点'].str.contains('TIKTOK')]
# 去掉TK仓
product_df = product_df[~product_df['仓库'].str.contains('TK')]

# 去除三方仓数据
product_df = product_df[~product_df['仓库'].str.contains('GA')]

# 去除异常仓 todo
product_df = product_df[~product_df['仓库'].str.contains('异常')]
'''
# 取需要的仓
product_df = product_df[product_df['仓库'].str.contains('广州仓|海外仓|易速达美东GA仓|B端|TK本地仓|顺丰SF')]

# 格式化站点 shop:site
product_df['店铺'] = product_df.站点.apply(lambda x: x if x == '共享' else x[len('AMAZON:'):])

# 构建 站点
product_df['站点'] = product_df.站点.apply(lambda x: x.split(':')[-1] if '共享' not in x else x)


In [ ]:
product_df.店铺.unique()

In [ ]:
product_df.仓库.unique()

In [ ]:
product_df.站点.unique()

In [ ]:
product_df.店铺.unique()

In [ ]:
# # 使用正则表达式一次性排除 '海外' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
# pattern = r'^(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓)(?!.*TK本地仓).*'
# local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
# local_inv.仓库.unique()

In [ ]:
# 使用正则表达式一次性排除 '海外'，'易速达美东GA仓'，'顺丰SF' 和 'B端'；只包含本地仓（仓库 值只包含本地仓的行）
pattern = r'^(?!.*顺丰SF)(?!.*海外)(?!.*B端)(?!.*易速达美东GA仓).*'
local_inv = product_df[pd.Series(product_df['仓库']).str.contains(pattern, regex=True)]
local_inv.仓库.unique()

In [ ]:
local_inv

In [ ]:
# local_inv

In [ ]:
local_inv.query('SKU == "SP001-406 Black 40"')

In [ ]:
# local_inv.query('SKU == "SP001-406 Black 40"')

In [ ]:
# 只包含海外仓（仓库 值只包含海外仓 的行）
sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓|顺丰SF')]
sea_inv.仓库.unique()

In [ ]:
# # 只包含海外仓（仓库 值只包含海外仓 的行）
# sea_inv = product_df[product_df.仓库.str.contains('海外|易速达美东GA仓|TK本地仓')]
# sea_inv.仓库.unique()

In [ ]:
sea_inv

# 3、预处理预占单据

## ①处理'已拣货'、'未出库'调拨单据(暂时未用到这部分数据——调拨单)

In [ ]:
condition = (
    (yuzhan_df['单据状态'] == '审核通过') &
    (yuzhan_df['拣货状态'] == '已拣货') &
    (yuzhan_df['出库状态'] == '未出库')
)

yuzhan_df = yuzhan_df.loc[condition]

# 区分FBA和海外仓预占
condition = yuzhan_df['调入仓'].str.contains('FBA')

# FBA仓预占
yuzhan_fba = yuzhan_df[condition]

# 海外仓预占
yuzhan_sea = yuzhan_df[~condition]


yuzhan_fba['仓库'] = yuzhan_fba['调入仓'].str.split(pat='_FBA', expand=True)[0]
yuzhan_fba['仓库'] = yuzhan_fba['仓库'].apply(lambda x: x.replace('EU', 'DE') if 'EU' in x else x)

yuzhan_fba = yuzhan_fba.pivot_table(index=['仓库','MSKU'], values=['调出数量'], aggfunc=sum).reset_index().rename(columns={'调出数量': 'FBA预占数'})

In [ ]:
yuzhan_df

In [ ]:
yuzhan_fba

In [ ]:
yuzhan_fba.columns

## ②处理'未出库'调拨单单据

In [ ]:
yiyuzhan_df = pd.read_excel(yiyuzhan_path, sheet_name = '明细数据', usecols=['SKU', 'MSKU', '调出仓', '调入仓', '调出数量', '物流方式', '单据状态', '拣货状态', '出库状态'])
# 将具体的渠道 映射为大渠道：快递、空运、海运
yiyuzhan_df = pd.merge(yiyuzhan_df, LogisticsMethod_map, how = 'left', on = '物流方式')
yiyuzhan_df.drop(columns=['物流方式'], inplace=True)
yiyuzhan_df.rename(columns={'大渠道': '物流方式'}, inplace=True)

# 确保每一个调拨单都能匹配上-物流方式
is_nan = yiyuzhan_df['物流方式'].isna()
if is_nan.sum() > 0:
    raise ValueError("缺失对于大渠道，请更新【物流渠道映射】子表")

In [ ]:
yiyuzhan_df['物流方式'].unique()

In [ ]:
condition = (
    (yiyuzhan_df['单据状态'] == '审核通过') &
    # (yiyuzhan_df['拣货状态'] == '未拣货') &
    (yiyuzhan_df['出库状态'] == '未出库')
)

yiyuzhan_df = yiyuzhan_df.loc[condition]

yiyuzhan_df['仓库'] = yiyuzhan_df['调入仓'].str.split(pat='_FBA', expand=True)[0]
yiyuzhan_df['仓库'] = yiyuzhan_df['仓库'].apply(lambda x: x.replace('EU', 'DE') if 'EU' in x else x)

In [ ]:
yiyuzhan_df['仓库']

In [ ]:
# 在 pivot_table 之后，手动补全缺失的物流方式列，确保每次结果都有这三列'快递', '空运', '海运'
def pivot_with_all_logistics(df):
    pivot = df.pivot_table(
        index=['仓库', 'MSKU'],
        columns='物流方式',
        values='调出数量',
        aggfunc='sum',
        fill_value=0
    ).reset_index()

    pivot.columns.name = None
    for col in ['快递', '空运', '海运', 'FBA自提物流']:
        if col not in pivot.columns:
            pivot[col] = 0
    # return pivot[['仓库', 'MSKU', 'FBA自提物流', '快递', '空运', '海运']]
    return pivot[['仓库', 'MSKU', '快递', '空运', '海运', 'FBA自提物流']]
yiyuzhan_pivot = pivot_with_all_logistics(yiyuzhan_df)

In [ ]:
# yiyuzhan_pivot.rename(columns={'FBA自提物流': 'FBA自提物流_预占','快递': '快递_预占','空运': '空运_预占','海运': '海运_预占'}, inplace=True)
yiyuzhan_pivot.rename(columns={'快递': '快递_预占','空运': '空运_预占','海运': '海运_预占','FBA自提物流':'FBA自提物流_预占'}, inplace=True)


In [ ]:
yiyuzhan_pivot.columns

In [ ]:
yiyuzhan_pivot

# 4、合并数据

In [ ]:
local_inv

## 4.1 匹配本地锁仓库存和FBA预占数

In [ ]:
 # 筛选出一个不包含 “共享” 店铺的数据框，也就是筛选出‘本地锁仓数’
local_inv_lock = local_inv[~local_inv['店铺'].str.contains('共享')]

# 按照 “店铺”、“SKU” 和 “MSKU” 分组，并计算出每个分组 “本地” 列总和的新数据框
local_inv_lock_piv = local_inv_lock.pivot_table(index=['店铺', 'SKU','MSKU'], values='本地', aggfunc='sum').reset_index()

# FBA库存匹配本地锁仓库存（FBA库存为主表）
new_inv = pd.merge(left=fba_process, right=local_inv_lock_piv, left_on=['仓库', 'SKU','MSKU'], right_on=['店铺', 'SKU','MSKU'], how='left')
new_inv.drop(columns=['店铺'], inplace=True)
new_inv.rename(columns={'本地': '本地锁仓数'}, inplace=True)     #'可用量' -> '本地' -> '本地锁仓数'

# 匹配FBA预占数
new_inv = pd.merge(left=new_inv, right=yuzhan_fba, left_on=['仓库', 'MSKU'], right_on=['仓库', 'MSKU'], how='left')

# 匹配FBA未拣货已预占数
new_inv = pd.merge(left=new_inv, right=yiyuzhan_pivot, left_on=['仓库', 'MSKU'], right_on=['仓库', 'MSKU'], how='left')


In [ ]:
new_inv

In [ ]:
local_inv_lock_piv

In [ ]:
local_inv_lock

In [ ]:
new_inv.head(10)

In [ ]:
# local_inv_lock.to_excel('店铺不包含共享数.xlsx',index = False)

In [ ]:
# local_inv_lock_piv.to_excel('不包含共享数.xlsx',index = False)

In [ ]:
fba_process.columns

In [ ]:
local_inv_lock_piv.query('MSKU == "SP001-406 Black 40"')

In [ ]:
local_inv_lock_piv

In [ ]:
new_inv.query('SKU == "SP001-415 Dark Grey 47"')

In [ ]:
new_inv.query('SKU == "SP001-415 Dark Grey 47"')

In [ ]:
new_inv.columns

In [ ]:
new_inv.columns

In [ ]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

In [ ]:
new_inv.query('MSKU == "SP001-415 Dark Grey 47"')

In [ ]:
local_inv_lock.query('MSKU == "SP001-415 Dark Grey 47"')

## 4.2 匹配本地共享数据


In [ ]:
# 采购数据，按照 “SKU”分组，并计算出每个分组 '已下单数量', '本地-在途' 列总和的新数据框。'本地-在途'（待质检数）
# todo
procurement_documents = local_inv.groupby(by=['店铺','SKU','MSKU'])[['已下单数量', '本地-在途', '预占总数', '已生产未发货']].sum().reset_index()
# procurement_documents = local_inv.groupby(by=['SKU'])[['已下单数量', '本地-在途']].sum().reset_index()

# 本地-共享
share_local = local_inv[local_inv['店铺'] == '共享'].groupby(by=['SKU'])['本地'].sum().reset_index().rename(columns={'本地': '本地-共享'})

In [ ]:
local_inv.columns

In [ ]:
procurement_documents.query('SKU == "SP001-415 Dark Grey 47"')

In [ ]:
share_local.query('SKU == "SP001-415 Dark Grey 47"')

In [ ]:
procurement_documents.rename(columns={'店铺':'仓库'}, inplace=True)

procurement_documents.columns

In [ ]:
procurement_documents.仓库.unique()

In [ ]:
dfs = [new_inv, share_local]

# 使用 reduce 函数进行左连接
merged_df = reduce(lambda left, right: pd.merge(left, right, on='SKU', how='left'), dfs)

In [ ]:
new_inv.columns

In [ ]:
new_inv.仓库.unique()   # 此数据框的仓库实际是店铺：站点

In [ ]:
local_inv.店铺.unique() 

In [ ]:
local_inv.仓库.unique() 

In [ ]:
procurement_documents.columns

In [ ]:
share_local.columns

In [ ]:
merged_df.query('SKU == "SP001-415 Dark Grey 47"')

In [ ]:
merged_df = pd.merge(merged_df, procurement_documents, on = ['仓库', 'SKU', 'MSKU'], how = 'left')

In [ ]:
merged_df.query('SKU == "SK001-701 Stripe Black 44-45"')

# 4.3 数据透视本地库存

In [ ]:
# product_local = product_df.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()
product_local = local_inv.pivot_table(index='SKU', columns='站点', values=['本地'], aggfunc=sum, fill_value=0).reset_index()


piv_df1 = pd.concat(objs=[product_local['SKU'], product_local['本地']], axis=1)
piv_df2 = local_inv.pivot_table(index='SKU', values=['本地-在途'], aggfunc=sum, fill_value=0).reset_index()

In [ ]:
piv_df2.columns

In [ ]:
local_piv = pd.merge(left=piv_df1, right=piv_df2, on='SKU', how='left')

In [ ]:
local_piv.query('SKU == "SP001-406 Black 40"')

In [ ]:
local_piv

# 5、处理海外仓库存

In [ ]:
# 构建海外仓站点映射
site_map = {'元坤海外仓': 'US', '九方欧洲海外仓': 'DE', 'CA仓搜海外仓' : 'CA', 
            'DE延讯海外仓':'DE', 'UK延讯海外仓' : 'UK', 'JP永翔海外仓' : 'JP',
            'DE商易海外仓' : 'DE', 'US商易海外仓' : 'US', 'US易速达海外仓': 'US', 
            'UK商易海外仓' : 'UK', 'CN易速达:易速达美东GA仓':'US', 'IT永翔海外仓':'IT',
            # '顺丰SF:美国特拉华S2仓' : 'US', '顺丰SF:美国洛杉矶S5仓' : 'US', '顺丰SF:美国达拉斯C1仓': 'US', 
            '顺丰SF:美国特拉华S2仓' : 'US', '顺丰SF:美国洛杉矶S5仓' : 'US',
            # '顺丰SF:美国芝加哥C1仓' : 'US', '顺丰SF:美国芝加哥S06仓' : 'US'}
            '顺丰SF:美国芝加哥S06仓' : 'US'}

In [ ]:
# # 构建海外仓站点映射
# site_map = {'元坤海外仓': 'US', '九方欧洲海外仓': 'DE', 'CA仓搜海外仓' : 'CA', 
#             'DE延讯海外仓':'DE', 'UK延讯海外仓' : 'UK', 'JP永翔海外仓' : 'JP',
#             'DE商易海外仓' : 'DE', 'US商易海外仓' : 'US', 'US易速达海外仓': 'US', 
#             'UK商易海外仓' : 'UK', 'CN易速达:易速达美东GA仓':'US', 'IT永翔海外仓':'IT',
#             # '顺丰SF:美国特拉华S2仓' : 'US', '顺丰SF:美国洛杉矶S5仓' : 'US', '顺丰SF:美国达拉斯C1仓': 'US', 
#             '顺丰SF:美国特拉华S2仓' : 'US', '顺丰SF:美国洛杉矶S5仓' : 'US',
#             # '顺丰SF:美国芝加哥C1仓' : 'US', '顺丰SF:美国芝加哥S06仓' : 'US'}
#            '顺丰SF:美国芝加哥S06仓' : 'US'}

In [ ]:
sea_inv['仓库'].unique()

In [ ]:
len(site_map)

In [ ]:
site_map

In [ ]:
sea_inv['仓库'].unique()

In [ ]:
if len(site_map) != sea_inv['仓库'].unique().size:
    raise ValueError(f"海外仓数量有变动，请重新维护变量：site_map ")

In [ ]:
# 为海外仓库 映射站点
sea_inv['站点'] = sea_inv['仓库'].apply(site_map.get)

In [ ]:
# 海外仓库存全部归为共享
temp_df = sea_inv[['站点','SKU', '仓库', '本地']].pivot_table(index=['站点','SKU'], columns=['仓库'], aggfunc=sum).fillna(0).reset_index()
sea_inv = pd.concat(objs=[temp_df['站点'], temp_df['SKU'], temp_df['本地']], axis=1)


In [ ]:
# 预处理海外仓-预占数据
yuzhan_sea['站点'] = yuzhan_sea['调入仓'].apply(site_map.get)

yuzhan_sea = yuzhan_sea.pivot_table(index=['站点', 'SKU'], columns=['调入仓'], values='调出数量', aggfunc=sum).reset_index()

# 重命名字段
columns = yuzhan_sea.columns
new_columns = columns[:2].to_list() + [f'{name}_预占' for name in columns[2:]]
yuzhan_sea.columns = new_columns
if yuzhan_sea.size == 0:
    print('本周无海外仓预占数据')

sea_ret = pd.merge(left=sea_inv, right=yuzhan_sea, on=['站点', 'SKU'], how='outer')

# 填充空值
numeric_columns = sea_ret.select_dtypes(include='number').columns
sea_ret[numeric_columns] = sea_ret[numeric_columns].fillna(0)

# 最后的数据处理与输出

In [ ]:
merged_df.query('SKU == "SP001-406 Black 40"')

In [ ]:
merged_df.query('SKU == "SP001-406 Black 40"')

In [ ]:
# 填充缺失值
numeric_columns = merged_df.select_dtypes(include='number').columns

merged_df[numeric_columns] = merged_df[numeric_columns].fillna(0)

In [ ]:
merged_df['FBA在途'] = merged_df['FBA预占数'] + merged_df['已发货(数量)'] + merged_df['正在接收(数量)']

In [ ]:
merged_df.columns

In [ ]:
# 调整字段输出顺序
# new_order = ['站点', '仓库', 'SKU', 'MSKU', '预占总数', 'FBA自提物流_预占', '快递_预占', '空运_预占', '海运_预占',
#                 'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
#                 '预留-运营中心转运','预留-运营中心正在处理', 'FBA可售', '可用库存', 
#                 '长期仓储库存', 
#                 '本地锁仓数', '本地-共享', '已下单数量', '本地-在途',
#                 '7天日均', '15天日均', '30天日均']
new_order = ['站点', '仓库', 'SKU', 'MSKU', '预占总数', '快递_预占', '空运_预占', '海运_预占','FBA自提物流_预占',
                'FBA预占数', '已发货(数量)', '正在接收(数量)', 'FBA在途',
                '预留-运营中心转运','预留-运营中心正在处理', 'FBA可售', '可用库存', 
                '长期仓储库存', 
                '本地锁仓数', '本地-共享', '已下单数量', '本地-在途', '已生产未发货',
                '7天日均', '15天日均', '30天日均']
merged_df = merged_df[new_order]

merged_df.rename(columns={'可用库存': 'FBA可用库存'}, inplace=True)

In [ ]:
# 要将本地库存数-未拣货数（预占数），因为要“未拣货数（预占数）”视为在途数
# merged_df['本地锁仓数'] = merged_df['本地锁仓数'] - merged_df['预占总数']

In [ ]:
merged_df.columns

In [ ]:
# 指定要添加颜色的列
# columns_to_color = {'站点': 'FFF07C82', '仓库': 'FFF07C82', 'SKU': 'FFF07C82', 'MSKU': 'FFF07C82', 
#                     '预占总数': 'FFFCD5B4', 'FBA自提物流_预占': 'FFFCD5B4', '快递_预占': 'FFFCD5B4', '空运_预占': 'FFFCD5B4','海运_预占': 'FFFCD5B4', 
#                 'FBA预占数': 'FF40A070', '已发货(数量)': 'FF40A070', '正在接收(数量)': 'FF40A070', 'FBA在途': 'FF40A070',
#                 '预留-运营中心转运': 'FF9ABEAF','预留-运营中心正在处理': 'FF9ABEAF', 'FBA可售': 'FF9ABEAF', 'FBA可用库存': 'FF9ABEAF', 
#                 '长期仓储库存': 'FF9ABEAF', 
#                 '本地锁仓数': 'FFE2D849', '本地-共享': 'FFE2D849', '已下单数量': 'FFE2D849', '本地-在途': 'FFE2D849',
#                 '7天日均': 'FFE16723', '15天日均': 'FFE16723', '30天日均': 'FFE16723'}
columns_to_color = {'站点': 'FFF07C82', '仓库': 'FFF07C82', 'SKU': 'FFF07C82', 'MSKU': 'FFF07C82', 
                    '预占总数': 'FFFCD5B4', '快递_预占': 'FFFCD5B4', '空运_预占': 'FFFCD5B4','海运_预占': 'FFFCD5B4', 'FBA自提物流_预占': 'FFFCD5B4',
                'FBA预占数': 'FF40A070', '已发货(数量)': 'FF40A070', '正在接收(数量)': 'FF40A070', 'FBA在途': 'FF40A070',
                '预留-运营中心转运': 'FF9ABEAF','预留-运营中心正在处理': 'FF9ABEAF', 'FBA可售': 'FF9ABEAF', 'FBA可用库存': 'FF9ABEAF', 
                '长期仓储库存': 'FF9ABEAF', 
                '本地锁仓数': 'FFE2D849', '本地-共享': 'FFE2D849', '已下单数量': 'FFE2D849', '本地-在途': 'FFE2D849','已生产未发货': 'FFE2D849',
                '7天日均': 'FFE16723', '15天日均': 'FFE16723', '30天日均': 'FFE16723'}

# 将 DataFrame 转换为 Excel 格式，并创建一个 Pandas Excel writer
with pd.ExcelWriter(f'../src_data/处理后的库存/库存{date_parm}.xlsx', engine='openpyxl') as writer:
    merged_df.to_excel(writer, sheet_name='库存汇总', index=False)
    local_piv.to_excel(writer, sheet_name='本地仓库存透视', index=False)
    sea_ret.to_excel(writer, sheet_name='海外仓库存', index=False)
    

    # 获取 Pandas Excel writer 中的 workbook 和 worksheet
    workbook = writer.book
    worksheet = writer.sheets['库存汇总']

    # 使用 Styler 对象为指定列的 header 行添加样式
    rows = merged_df.shape[0]
    for column, color in columns_to_color.items():
        col_idx = merged_df.columns.get_loc(column) + 1
        # print(col_idx)
        for index in range(1, rows+2):
            header_cell = worksheet.cell(index, col_idx)
            header_cell.fill = openpyxl.styles.PatternFill(start_color=color, end_color=color, fill_type='solid')

# 打印结果
print("Excel 文件已生成")
